In [ ]:
# !df -h /kaggle/working

In [ ]:
# !du -sh /kaggle/working/* | sort -rh

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
# Prevention for memory fragmentation

In [ ]:
max_seq_length = 2048 # Adjust based on legal document lengths
dtype = None # Auto-detection
load_in_4bit = True # Essential for T4 GPUs

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters for DAPT
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Higher rank is often better for DAPT to absorb new knowledge
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset, load_from_disk

# Load your uploaded Parquet file
# dataset = load_dataset("parquet", data_files={"train": "/kaggle/input/datasets/rehanfargose/sc-dataset-1990-2025/SC_Parquet_Dataset_1990-2025.parquet"}, split="train")
dataset = load_from_disk("/kaggle/input/datasets/rehanfargose/tokenized-dataset")

In [ ]:
print(dataset[0].keys())
# Should show dict_keys(['input_ids', 'attention_mask'])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "input_ids", # We use the raw IDs directly
    max_seq_length = max_seq_length,
    packing = False, # CRITICAL: Set to False; already packed the data
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Total batch size of 4 across 2 GPUs
        gradient_accumulation_steps = 16, # Total effective batch of 16
        warmup_steps = 20,
        max_steps = 50, # Start with 500 to test, then increase
        learning_rate = 5e-5, # DAPT usually needs a lower LR than 2e-4
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Standard for dual T4
        weight_decay = 0.01,
        save_strategy = "no",
        report_to = "none",
        # Important for dual GPU stability:
        ddp_find_unused_parameters = False,
    ),
)

In [ ]:
trainer.train()

In [ ]:
# This saves ONLY the trained adapter layers (very small and disk-safe)
model.save_pretrained("Deepseek_R1_8B_DAPT2_LoRA")
tokenizer.save_pretrained("Deepseek_R1_8B_DAPT2_LoRA")

In [ ]:
import shutil
from IPython.display import FileLink

# Zip the small adapter folder
shutil.make_archive("Deepseek_R1_8B_DAPT2_LoRA", 'zip', "Deepseek_R1_8B_DAPT2_LoRA")

# Generate the download link
FileLink(r'Deepseek_R1_8B_DAPT2_LoRA.zip')